In [67]:
import numpy as np
import pandas as pd
import lseg.data as ld
import matplotlib.pyplot as plt
import time
from pathlib import Path

In [68]:
# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [69]:
# open Refinitiv session
ld.open_session()

<lseg.data.session.Definition object at 0x1dbf888b140 {name='workspace'}>

In [70]:
FRQ = {"daily": "D", "weekly": "W", "monthly": "M"}

JOBS = {
    "price":  {"field": "TR.PriceClose",                     "interval": "monthly", "batch_size": 50},
    "volume": {"field": "TR.Volume",                         "interval": "daily",   "batch_size": 10},
    "shares": {"field": "TR.TOTALNUMBEROFSHARESOUTSTANDING", "interval": "monthly", "batch_size": 50},
}

In [71]:
def to_long(df):
    shaped_df = df.stack(level=0)
    shaped_df = shaped_df.reset_index()
    shaped_df = shaped_df.rename(columns={"level_1": "ric", "Date": "date"})
    return shaped_df

In [72]:
# Create the directory name
def raw_dir(index_ric, start, end, label):
    name = f"{index_ric.lstrip('.')}_{start}_{end}_{label}"
    return Path("data/raw") / name

In [73]:
# Show stocks listed on an exchange each month (or whatever frequency is chosen)
def get_members(index_ric, start, end, freq="ME"):
    # 1. All joiners and leavers between start and end
    const_changes = ld.get_data(
        universe=[index_ric],
        fields=["TR.IndexJLConstituentChangeDate", "TR.IndexJLConstituentRIC", 
                "TR.IndexJLConstituentName", "TR.IndexJLConstituentituentChange"],
        parameters={"SDATE":f"{start}","EDATE":f"{end}", 'IC':'B'}
    )
    const_changes["Date"] = pd.to_datetime(const_changes["Date"])

    # 2. Members ON the start date: the members field, not the changes field
    start_members = ld.get_data(
        universe=[index_ric], 
        fields=["TR.IndexConstituentRIC"], 
        parameters={"SDATE":f"{start}"}
    )
    init_list = start_members["Constituent RIC"].dropna().tolist()

    # 3. Replay history month by month
    rows = []
    prev = pd.to_datetime(start)
    for m in pd.date_range(start, end, freq=freq):
        in_window = (const_changes["Date"] > prev) & (const_changes["Date"] <= m)
        window = const_changes[in_window]
        window = window.sort_values("Date")

        for ric, change in zip(window["Constituent RIC"], window["Change"]):
            if change == "Leaver" and ric in init_list:
                init_list.remove(ric)
            elif change == "Joiner" and ric not in init_list:
                init_list.append(ric)

        for ric in init_list:
            rows.append({"Date": m, "ric": ric})
        prev = m

    return pd.DataFrame(rows)

def members_path(index_ric, start, end):
    folder = raw_dir(index_ric, start, end, "members")
    folder.mkdir(parents=True, exist_ok=True)
    return folder / "members.parquet"


def save_members(members, index_ric, start, end):
    path = members_path(index_ric, start, end)
    members.rename(columns={"Date": "date"}).to_parquet(path)
    print(f"saved {path} ({len(members)} rows)")


def load_or_get_members(index_ric, start, end):
    path = members_path(index_ric, start, end)
    if path.exists():
        return pd.read_parquet(path)
    members = get_members(index_ric, start, end)
    save_members(members, index_ric, start, end)
    return members.rename(columns={"Date": "date"})

In [74]:
def fetch_with_retry(batch, field, interval, start, end, tries=3):
    for attempt in range(tries):
        try:
            return ld.get_data(
                universe=batch,
                fields=[f"{field}.date", field],
                parameters={"SDate": start, "EDate": end, "Frq": FRQ[interval]}
            )
        except Exception as e:
            print(f"attempt {attempt + 1} failed: {e}. Waiting {2 ** attempt}s")
            time.sleep(2 ** attempt)
    return None
        

def download_all(index_ric, rics, label, start, end, wait=0.3):
    job = JOBS[label]
    size = job["batch_size"]
    batches = [rics[i: i+size] for i in range(0, len(rics), size)]
    failed = []

    out_dir = raw_dir(index_ric, start, end, label)
    out_dir.mkdir(parents=True, exist_ok=True)

    for index, b in enumerate(batches):
        # Create the parquet filename
        path = out_dir / f"batch_{index:04d}.parquet"
        if path.exists(): 
            continue

        df = fetch_with_retry(b, job["field"], job["interval"], start, end)
        if df is None or df.empty:
            failed.append((index, b))
            continue

        df.columns = ["ric", "date", label]
        df["date"] = pd.to_datetime(df["date"])

        # Create a tmp file in case it gets interrupted
        tmp = path.with_suffix(".tmp")
        df.to_parquet(tmp)
        tmp.rename(path)
        print(f"saved {path.name} ({len(df)} rows)")
        
        # Pause to reduce requests to Refinitiv per second
        time.sleep(wait)

    return failed

In [75]:
def load_all(index_ric, start, end, label):
    folder = raw_dir(index_ric, start, end, label)
    tables = [pd.read_parquet(p) for p in sorted(folder.glob("*.parquet"))]
    if not tables:
        raise ValueError(f"no files in {folder}")
    panel = pd.concat(tables, ignore_index=True)
    return panel.sort_values(["ric", "date"]).reset_index(drop=True)

In [76]:
index_ric = ".NDX"
start = "2020-01-01"
end = "2026-01-01"

members = load_or_get_members(index_ric, start, end)
rics = sorted(members["ric"].unique().tolist())

for label in JOBS:
    failed = download_all(index_ric, rics, label, start, end)
    print(label, "failed:", failed)

price = load_all(index_ric, start, end, "price")
volume = load_all(index_ric, start, end, "volume")
shares = load_all(index_ric, start, end, "shares")

saved data\raw\NDX_2020-01-01_2026-01-01_members\members.parquet (7313 rows)
saved batch_0000.parquet (3600 rows)
saved batch_0001.parquet (3600 rows)
saved batch_0002.parquet (3600 rows)
saved batch_0003.parquet (288 rows)
price failed: []
saved batch_0000.parquet (15080 rows)
saved batch_0001.parquet (15080 rows)
saved batch_0002.parquet (15080 rows)
saved batch_0003.parquet (15080 rows)
saved batch_0004.parquet (15080 rows)
saved batch_0005.parquet (15080 rows)
saved batch_0006.parquet (15080 rows)
saved batch_0007.parquet (15080 rows)
saved batch_0008.parquet (15080 rows)
saved batch_0009.parquet (15080 rows)
saved batch_0010.parquet (15080 rows)
saved batch_0011.parquet (15080 rows)
saved batch_0012.parquet (15080 rows)
saved batch_0013.parquet (15080 rows)
saved batch_0014.parquet (15080 rows)
saved batch_0015.parquet (6032 rows)
volume failed: []
attempt 1 failed: Unable to collect data for the field 'TR.TOTALNUMBEROFSHARESOUTSTANDING.DATE' and some specific identifier(s). Reque